In [1]:
# 1. Put this repository first on sys.path, so `recipes` is its recipes/ package,
#    then construct the Cortex Training client from the connection file created during setup (~/cortex-training-config.json).
import json
import sys
from pathlib import Path

REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "recipes" / "utils.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter inside a cortex-training checkout so `recipes` can be imported.")
sys.path.insert(0, str(REPO_ROOT))
import recipes  # noqa: E402

if Path(recipes.__file__).resolve().parent != REPO_ROOT / "recipes":
    raise RuntimeError(
        f"`recipes` resolves to {recipes.__file__}, outside {REPO_ROOT}. Another package named "
        "`recipes` is installed; uninstall it or restart the kernel, then run this cell again."
    )
from recipes.utils import make_config  # noqa: E402
from recipes.utils import make_client  # noqa: E402

import os
config_path = Path(os.environ.get("CORTEX_TRAINING_CONFIG") or Path.home() / "cortex-training-config.json")
if not config_path.is_file():
    raise RuntimeError(
        f"Connection file not found at {config_path}. Create it as in docs/getting-started/setup.md "
        "(outside the repository), or set CORTEX_TRAINING_CONFIG to its path."
    )
# make_client reads the whole connection mapping -- host, PAT, database,
# schema, endpoint and options -- not just the token, so a configured file
# is honored in full.
client = make_client(str(config_path))

In [2]:
# 2. Build the Qwen3-8B sampling job from the model catalog. The override is for this
#    demo only: one GPU and a 2,048-token context instead of the catalog's settings.
job_body = make_config(
    model="Qwen/Qwen3-8B",
    context="standard",
    inference=True,
    overrides=[
        {
            "sub_job_configs": [
                {"job_type": "sampling", "inference_config": {"n_gpus": 1, "max_seq_len": 2048}}
            ]
        }
    ],
)

In [3]:
# 3. Submit the job and wait for the sampling worker to start (will take about 3 min).
job_id = client.create_job_from_body(job_body)["job_id"]
print(f"Job {job_id} is starting")
client.wait_for_job(job_id)
print(f"Job {job_id} is running")

Job 00000000-0000-0000-0000-000000000000 is starting
Job 00000000-0000-0000-0000-000000000000 is running


In [4]:
# 4. Generate text and wait for the asynchronous request to finish.
request_id = client.generate(
    job_id,
    prompts=["Explain in one paragraph why snowflakes have six-fold symmetry."],
    sampling_params={"max_tokens": 128, "temperature": 0.7},
)
result = client.poll_request(job_id, request_id)
print(result['results'][0]['text'])

 Snowflakes exhibit six-fold symmetry due to the molecular structure of water and the way it crystallizes. When water freezes, its molecules arrange themselves into a hexagonal lattice, a structure that is energetically favorable because it allows for optimal hydrogen bonding between water molecules. This hexagonal arrangement leads to the formation of a six-sided (hexagonal) crystal structure, which is why snowflakes typically display six-fold symmetry. As the snowflake grows, each of the six arms develops in a similar manner, influenced by temperature and humidity conditions, resulting in the intricate and symmetrical patterns observed in snowflakes. The underlying hexagonal symmetry


In [5]:
# 5. Tear down the sampling job and release its GPU.
client.cancel_job(job_id)
print(f"Cancelled job {job_id}")
job_id = None

Cancelled job 00000000-0000-0000-0000-000000000000
